In [ ]:
!wget -q https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt -O tinyshakespeare.txt
!head -100 tinyshakespeare.txt

In [ ]:
from torch import nn

class Encoder:

    def __init__(self, path):
        with open(path) as f:
            lines = f.readlines()

        text = "\n".join(lines)

        index = 1
        self.dictionary = {"⒨": 0}
        for char in text:
            if char in self.dictionary:
                continue

            self.dictionary[char] = index
            index += 1

    def encode(self, text):
        return [self.dictionary[char] for char in text]

    def decode(self, encoded) -> list[int]:
        inv_dict = {v: k for k, v in self.dictionary.items()}
        return [inv_dict[encoding] for encoding in encoded]

    def vocab(self):
        return self.dictionary.keys()

encoder = Encoder("tinyshakespeare.txt")
encoded = encoder.encode("Hello World!")
print(encoded)
decoded = encoder.decode(encoded)
print(decoded)


In [ ]:
from torch import nn, Tensor, softmax, randn

class Transformer(nn.Module):

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        self.encoder = Encoder("tinyshakespeare.txt")

        vocab_size = len(self.encoder.vocab())
        max_seq_len = 1024
        embed_dim = 256
        hidden_dim = 4 * embed_dim
        num_layers = 16

        self.token_emb = nn.Parameter(randn(vocab_size, embed_dim))
        self.pos_emb = nn.Parameter(randn(max_seq_len, embed_dim))
        self.mask_prob_emb = nn.Linear(1, embed_dim)

        self.layers = nn.ModuleList([
            nn.ModuleList([
                nn.LayerNorm(embed_dim),
                nn.Linear(embed_dim, embed_dim), #W_Q
                nn.Linear(embed_dim, embed_dim), #W_K
                nn.Linear(embed_dim, embed_dim), #W_V

                #FFN
                nn.LayerNorm(embed_dim),
                nn.Linear(embed_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, embed_dim)])
            for _ in range(num_layers)
        ])

        self.output = nn.Linear(embed_dim, vocab_size)

    def embedding(self, x: Tensor, mask_prob: Tensor):
        t_emb = self.mask_prob_emb(mask_prob)
        return self.token_emb[x] + self.pos_emb[:x.shape[-1]] + t_emb

    def attention(self, Q: Tensor, K: Tensor, V: Tensor):
        d_k = K.shape[-1]
        return softmax(Q @ K.transpose(-2, -1) / (d_k ** 0.5), dim=-1) @ V

    def forward(self, x: Tensor, mask_prob: Tensor) -> Tensor:
        x = self.embedding(x, mask_prob)
        for ln1, W_Q, W_K, W_V, ln2, linear1, relu, linear2 in self.layers:
            x = x + self.attention(W_Q(ln1(x)), W_K(ln1(x)), W_V(ln1(x)))
            x = x + linear2(relu(linear1(ln2(x))))

        return self.output(x)

In [ ]:
import random
import torch
import time
import ipywidgets as widgets
from IPython.display import display

mask_index = 0

def sample(model, query, length, device, total_steps=20):

    # Widget for printing 
    out = widgets.Output()
    display(out)

    # Tokenize text input
    tokens = model.encoder.encode(query)
    x = torch.full((length,), mask_index, dtype=torch.long, device=device)
    x[:len(tokens)] = torch.tensor(tokens, device=device)
    fixed = (x != mask_index)

    with torch.no_grad():
        for step in range(total_steps):
            # Predict probabilities for each token at each position
            mask_prob = torch.tensor([1.0 - step / total_steps], device=device)
            predictions = model.forward(x, mask_prob)
            predictions[..., mask_index] = -float("inf")
            probs = torch.softmax(predictions, dim=-1)

            mask_positions = (x == mask_index) & ~fixed
            if not mask_positions.any():
                break

            for pos in mask_positions.nonzero(as_tuple=True)[0]:
                if random.random() < 1 / (total_steps - step):
                    x[pos] = torch.multinomial(probs[pos], 1).item()
            
            # Print out the generated text
            with out:
                out.clear_output(wait=True)
                print(''.join(model.encoder.decode(x.tolist())))


## Training

1. Sample a chunk of text data: $x_0 \sim$ TinyShakespeare.
2. Sample diffusion time $t \sim Uniform(0,1)$.
3. Mask each token with probability $t$.
4. Run the diffusion model on the masked tokens and $t$.
5. Train only masked positions with the MDLM weighted masked-token loss, using weight $1/t$ for the linear schedule.
6. Backprop.


In [ ]:
import math
import torch
import random
import time
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open("tinyshakespeare.txt") as f:
    text = f.read()

split = int(0.9 * len(text))
train_text, val_text = text[:split], text[split:]

seq_len = 128
batch_size = 64
iterations = 100000
checkpoint_every = 10000

transformer = Transformer().to(device)
optimizer = torch.optim.Adam(transformer.parameters(), lr=1e-4)

def grab_chunk(src) -> torch.Tensor:
    start = random.randint(0, len(src) - seq_len)
    return torch.tensor(transformer.encoder.encode(src[start:start + seq_len]), device=device)

def add_noise(x, t):
    mask = (torch.rand_like(x, dtype=torch.float) < t).long()
    return x * (1 - mask), mask

In [ ]:
for i in tqdm(range(iterations)):
    batch = torch.stack([grab_chunk(train_text) for _ in range(batch_size)])
    t = random.uniform(1e-3, 1)
    masked, mask = add_noise(batch, t)
    if not mask.any():
        continue

    preds = transformer(masked, torch.tensor([t], device=device))
    ce = torch.nn.functional.cross_entropy(preds[mask == 1], batch[mask == 1])
    loss = ce / t

    if i % checkpoint_every == 0:
        sample(transformer, "To be, ", 64, device)
    
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

# Save final model
sample(transformer, "To be, ", 64, device)
torch.save(transformer.state_dict(), f"mdlm.pt")


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
transformer = Transformer().to(device)
transformer.load_state_dict(torch.load("mdlm.pt", map_location=device))


In [ ]:
sample(transformer, "To be, ", 128, device, 1000)

In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters())

count_params(transformer)